# How do models find structure without labels?
**Topics:** K-Means · DBSCAN · Hierarchical Clustering · Gaussian Mixture Models


---
## 1. K-Means Clustering

### What it is
- Partitions data into K clusters by minimizing within-cluster variance
- Each point belongs to the cluster with the nearest centroid
- Iterates: assign points → update centroids → repeat until stable

### Key intuition
- Place K centroids randomly → assign each point to nearest centroid → move centroid to mean of its points → repeat
- Converges when assignments stop changing

### When to use
- Data has roughly spherical, similarly-sized clusters
- K is known or can be estimated
- Large datasets — computationally efficient

### When not to use
- Clusters are non-spherical or have very different sizes/densities
- K is unknown and hard to estimate
- Data has many outliers — centroids are sensitive to them


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# ── From scratch ──
def kmeans_scratch(X, k, max_iter=100, seed=42):
    rng = np.random.default_rng(seed)
    centroids = X[rng.choice(len(X), k, replace=False)]
    for _ in range(max_iter):
        distances = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids
    return labels, centroids

labels_scratch, centroids = kmeans_scratch(X, k=4)
print(f"From scratch — unique labels: {np.unique(labels_scratch)}")

# ── scikit-learn ──
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X)
print(f"Inertia (within-cluster variance): {kmeans.inertia_:.2f}")
print(f"Silhouette score                 : {silhouette_score(X, kmeans.labels_):.3f}")

# Choosing K — elbow method
inertias = []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
print(f"\nInertia by K: {dict(zip(range(2,10), [round(i,0) for i in inertias]))}")


### Interview Q&A

**How do you choose K?**
- Elbow method → plot inertia vs K, pick the "elbow" where gains diminish
- Silhouette score → measures how well points fit their cluster vs others (higher = better, range -1 to 1)

**What is inertia?**
- Sum of squared distances from each point to its cluster centroid
- Lower = tighter clusters, but always decreases as K increases

**What are the limitations of K-means?**
- Assumes spherical clusters of similar size
- Sensitive to initialization — use `n_init=10` to run multiple times
- Sensitive to outliers — consider K-medoids instead

### Gotchas
- Always standardize features before K-means — distance-based, scale matters
- Random initialization can give poor results — `KMeans` uses k-means++ by default (smarter init)
- Inertia alone is not enough to evaluate — always check silhouette score too


---
## 2. DBSCAN

### What it is
- Density-Based Spatial Clustering of Applications with Noise
- Groups points that are closely packed, marks sparse points as outliers (-1)
- Does not require specifying K upfront

### Key intuition
- A point is a **core point** if it has at least `min_samples` neighbors within radius `eps`
- Core points form clusters by connecting to nearby core points
- Points not reachable from any core point → noise/outliers

### When to use
- Clusters have irregular shapes (non-spherical)
- Number of clusters is unknown
- Dataset contains outliers/noise you want to detect
- Geospatial data, anomaly detection

### When not to use
- Clusters have very different densities — single eps won't work for all
- High-dimensional data — distances become meaningless (curse of dimensionality)
- Very large datasets — can be slow without spatial indexing


In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons, make_blobs
from sklearn.preprocessing import StandardScaler

# make_moons — non-spherical clusters that K-means fails on
X, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
X = StandardScaler().fit_transform(X)

db = DBSCAN(eps=0.3, min_samples=5)
labels = db.fit_predict(X)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()

print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {n_noise}")
print(f"Labels         : {np.unique(labels)}")

# Tuning eps — use k-distance graph
from sklearn.neighbors import NearestNeighbors

k = 5
nbrs = NearestNeighbors(n_neighbors=k).fit(X)
distances, _ = nbrs.kneighbors(X)
kth_distances = np.sort(distances[:, k-1])[::-1]
print(f"\nk-distance range: {kth_distances.min():.3f} to {kth_distances.max():.3f}")
print("Plot kth_distances to find the 'elbow' — set that as eps")


### Interview Q&A

**How do you tune eps and min_samples?**
- Plot k-distance graph (sorted distances to k-th nearest neighbor) — look for elbow
- min_samples ≥ dimensionality + 1 is a common rule of thumb

**DBSCAN vs K-means?**
- DBSCAN → no K needed, handles noise, finds arbitrary shapes
- K-means → faster, works better when clusters are spherical and similar in size

**What are core, border, and noise points?**
- Core → has min_samples neighbors within eps
- Border → within eps of a core point but fewer than min_samples own neighbors
- Noise → not within eps of any core point → labeled -1

### Gotchas
- DBSCAN struggles when clusters have very different densities — try HDBSCAN instead
- Scale features before DBSCAN — eps is distance-based
- Label -1 means noise/outlier, not a cluster


---
## 3. Hierarchical Clustering

### What it is
- Builds a tree (dendrogram) of clusters by progressively merging (agglomerative) or splitting (divisive) groups
- Agglomerative (bottom-up) is most common: start with each point as its own cluster → merge closest pairs

### Key intuition
- Like building a family tree for your data
- No need to specify K upfront — cut the dendrogram at any height to get any number of clusters

### Linkage methods

| Linkage | Merges clusters by | Best for |
|---|---|---|
| Single | Minimum distance between points | Elongated clusters |
| Complete | Maximum distance between points | Compact clusters |
| Average | Mean distance between all pairs | General use |
| Ward | Minimizes within-cluster variance | Most common default |

### When to use
- Want to explore cluster structure at multiple granularities
- Small to medium datasets (dendrograms get unreadable above ~1000 samples)
- Hierarchical relationships make sense (taxonomy, documents)

### When not to use
- Large datasets — O(n²) memory and time complexity
- Need to assign new data points to clusters — no predict() method


In [ ]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score

X, y_true = make_blobs(n_samples=150, centers=3, random_state=42)

# ── scikit-learn ──
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels = agg.fit_predict(X)
print(f"Silhouette score (ward, k=3): {silhouette_score(X, labels):.3f}")

# Compare linkage methods
for linkage_method in ['ward', 'complete', 'average', 'single']:
    model = AgglomerativeClustering(n_clusters=3, linkage=linkage_method)
    lbl = model.fit_predict(X)
    score = silhouette_score(X, lbl)
    print(f"Linkage: {linkage_method:<10} Silhouette: {score:.3f}")

# ── Dendrogram (scipy) ──
Z = linkage(X[:50], method='ward')   # subset for readability
print(f"\nDendrogram linkage matrix shape: {Z.shape}")
print("Each row: [cluster1, cluster2, distance, size]")
print(f"First merge at distance: {Z[0, 2]:.3f}")
print(f"Last merge at distance : {Z[-1, 2]:.3f}")


### Interview Q&A

**How do you decide where to cut the dendrogram?**
- Look for the largest vertical gap in the dendrogram — cut there
- Or use silhouette score across different K values

**Agglomerative vs divisive?**
- Agglomerative (bottom-up) → start with N clusters, merge to 1. Most common, easier to implement
- Divisive (top-down) → start with 1 cluster, split to N. Less common, more expensive

**Why Ward linkage is usually preferred?**
- Minimizes within-cluster variance at each merge — tends to produce compact, evenly sized clusters

### Gotchas
- No predict() for new data — if you need that, use K-means initialized from hierarchical results
- Ward linkage only works with Euclidean distance
- Dendrograms become unreadable with many samples — use on subsets for exploration


---
## 4. Gaussian Mixture Models (GMM)

### What it is
- Probabilistic model that assumes data is generated from a mixture of K Gaussian distributions
- Assigns each point a **probability** of belonging to each cluster (soft assignment)
- Trained using the Expectation-Maximization (EM) algorithm

### Key intuition
- K-means gives hard labels (you belong to cluster 1). GMM gives probabilities (70% cluster 1, 30% cluster 2)
- Models each cluster as an ellipse (Gaussian) — more flexible than K-means spheres
- EM alternates: estimate cluster memberships (E-step) → update Gaussian parameters (M-step)

### When to use
- Clusters overlap and you want probabilistic assignments
- Clusters have different shapes, sizes, or orientations (use covariance_type='full')
- Density estimation — model the underlying data distribution

### When not to use
- Very large datasets — EM is slower than K-means
- Hard cluster assignments are needed — use K-means
- Many dimensions without enough data — covariance estimation becomes unreliable


In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=[1.0, 1.5, 0.5], random_state=42)

# ── GMM ──
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm.fit(X)

labels      = gmm.predict(X)           # hard labels
probs       = gmm.predict_proba(X)     # soft probabilities
log_lik     = gmm.score(X)             # mean log-likelihood

print(f"Silhouette score  : {silhouette_score(X, labels):.3f}")
print(f"Mean log-likelihood: {log_lik:.3f}")
print(f"\nSample probabilities (first 5 rows):")
print(probs[:5].round(3))

# Choosing K — use BIC (lower = better)
bics = []
for k in range(2, 8):
    g = GaussianMixture(n_components=k, random_state=42)
    g.fit(X)
    bics.append((k, round(g.bic(X), 1)))
print(f"\nBIC by K: {bics}")
print("Pick K with lowest BIC")

# Covariance types
for cov in ['spherical', 'diag', 'tied', 'full']:
    g = GaussianMixture(n_components=3, covariance_type=cov, random_state=42)
    g.fit(X)
    print(f"covariance_type={cov:<12} BIC: {g.bic(X):.1f}")


### Interview Q&A

**GMM vs K-means?**
- K-means → hard assignments, assumes spherical equal-size clusters, faster
- GMM → soft probabilities, flexible cluster shapes, slower, works better on overlapping clusters

**What is the EM algorithm?**
- E-step: compute probability each point belongs to each Gaussian (soft assignment)
- M-step: update Gaussian parameters (mean, covariance) using weighted points
- Repeat until convergence

**How do you choose K in GMM?**
- Use BIC (Bayesian Information Criterion) or AIC — lower is better
- BIC penalizes complexity more than AIC → tends to pick simpler models

**What are covariance types?**
- spherical → clusters are circles, all same size
- diag → clusters are axis-aligned ellipses
- tied → clusters share the same shape
- full → each cluster has its own arbitrary ellipse (most flexible, most parameters)

### Gotchas
- GMM can converge to poor local optima — use `n_init > 1`
- Full covariance needs many samples per cluster — otherwise ill-conditioned
- EM is not guaranteed to find global optimum — always check log-likelihood
